include(joinpath(@__DIR__, "..", "..", "src", "paths.jl"))
# Exhaustive

In [1]:
ENV["GKSwstype"] = "100"
using Combinatorics
using Random
using LinearAlgebra
using Statistics

function calculate_MFPT(b::Vector{Float64})
    N_minus_1 = length(b)
    tau = zeros(Float64, N_minus_1)
    tau[1] = 1.0 / b[1]
    for m in 2:N_minus_1
        x_m = (1.0 - b[m]) / b[m]
        tau[m] = (1.0 / b[m]) + x_m * tau[m-1]
    end
    return sum(tau)
end

function calculate_second_moment(b::Vector{Float64}, d::Vector{Float64})
    n_minus_1 = length(b)
    ratios = d ./ b

    inner_sums = zeros(Float64, n_minus_1)
    inner_sums[1] = 1.0 / b[1]
    for k in 2:n_minus_1
        inner_sums[k] = 1.0 / b[k] + ratios[k] * inner_sums[k - 1]
    end

    suffix_sums = zeros(Float64, n_minus_1 + 1)
    for index in n_minus_1:-1:1
        suffix_sums[index] = suffix_sums[index + 1] + inner_sums[index]
    end

    second_sum = 0.0
    for m in 1:n_minus_1
        for i in 1:m
            product = 1.0
            for j in (i + 1):m
                product *= ratios[j]
            end
            second_sum += (1.0 / b[i]) * suffix_sums[i] * product
        end
    end

    return 2.0 * second_sum - calculate_MFPT(b)
end

function extract_parameters_for_N(N::Int, seed::Int=42)
    Random.seed!(seed)
    base_b = rand(N - 1)
    
    all_perms = collect(permutations(1:(N-1)))
    
    mfpt_values_all = Float64[]
    variance_values_all = Float64[]
    
    for p in all_perms
        b = base_b[p]
        mfpt = calculate_MFPT(b)
        second_moment = calculate_second_moment(b, 1.0 .- b)
        push!(mfpt_values_all, mfpt)
        push!(variance_values_all, second_moment - mfpt^2)
    end
    
    num_bins = 50
    log_mfpt_vals = log10.(mfpt_values_all)
    min_log_mfpt = minimum(log_mfpt_vals)
    max_log_mfpt = maximum(log_mfpt_vals)
    
    bins = range(min_log_mfpt, max_log_mfpt, length=num_bins+1)
    
    bin_centers = Float64[]
    upper_deltas = Float64[]
    lower_deltas = Float64[]
    med_vars = Float64[]

    for i in 1:num_bins
        if i == num_bins
            in_bin = (log_mfpt_vals .>= bins[i]) .& (log_mfpt_vals .<= bins[i+1])
        else
            in_bin = (log_mfpt_vals .>= bins[i]) .& (log_mfpt_vals .< bins[i+1])
        end
        
        if sum(in_bin) > 5
            vars_in_bin = variance_values_all[in_bin]
            max_var = maximum(vars_in_bin)
            min_var = minimum(vars_in_bin)
            med_var = median(vars_in_bin)
            
            push!(bin_centers, 10^((bins[i] + bins[i+1]) / 2.0))
            push!(upper_deltas, max_var - med_var)
            push!(lower_deltas, med_var - min_var)
            push!(med_vars, med_var)
        end
    end

    mid_log_mfpt = (min_log_mfpt + max_log_mfpt) / 2.0
    valid_indices = findall(i -> upper_deltas[i] > 0 && log10(bin_centers[i]) >= mid_log_mfpt, 1:length(bin_centers))
    
    T1_centers_fit = bin_centers[valid_indices]
    log_T1_fit = log10.(T1_centers_fit)
    log_Vmed_fit = log10.(med_vars[valid_indices])
    
    symmetric_spreads_fit = (upper_deltas[valid_indices] .+ lower_deltas[valid_indices]) ./ 2.0
    log_spread_fit = log10.(symmetric_spreads_fit)
    
    X_fit = hcat(ones(length(log_T1_fit)), log_T1_fit)
    
    params_med = X_fit \ log_Vmed_fit
    A = 10^params_med[1]
    gamma = params_med[2]
    
    params_spread = X_fit \ log_spread_fit
    B = 10^params_spread[1]
    delta = params_spread[2]
    
    return A, gamma, B, delta
end

function derive_generalized_phenomenology(N_range)
    As = Float64[]
    Bs = Float64[]
    gammas = Float64[]
    deltas = Float64[]
    
    println("--- EXTRACTING PARAMETERS ACROSS SIZES ---")
    for N in N_range
        println("Analyzing N = $N...")
        A, gamma, B, delta = extract_parameters_for_N(N)
        push!(As, A)
        push!(Bs, B)
        push!(gammas, gamma)
        push!(deltas, delta)
    end
    
    # 1. Check if exponents gamma and delta are roughly constant across N
    mean_gamma = mean(gammas)
    mean_delta = mean(deltas)
    
    # 2. Fit the prefactors A and B as power laws of N: A(N) = A0 * N^alpha
    log_Ns = log10.(collect(N_range))
    X_N = hcat(ones(length(log_Ns)), log_Ns)
    
    params_A = X_N \ log10.(As)
    A0 = 10^params_A[1]
    alpha = params_A[2]
    
    params_B = X_N \ log10.(Bs)
    B0 = 10^params_B[1]
    beta = params_B[2]
    
    println("\n=== GENERALIZED N-DEPENDENT PHENOMENOLOGICAL RELATION ===")
    println("Averaged Time Exponents (expected ~2.0):")
    println("  γ (gamma) = $(round(mean_gamma, digits=4))")
    println("  δ (delta) = $(round(mean_delta, digits=4))")
    
    println("\nScaling of Prefactors with System Size N:")
    println("  A(N) = $(round(A0, digits=4)) * N ^ $(round(alpha, digits=4))")
    println("  B(N) = $(round(B0, digits=4)) * N ^ $(round(beta, digits=4))")
    
    println("\nGeneral Phenomenological Bounds for any N:")
    println("  Var_med(T1, N) ≈ ($(round(A0, digits=4)) * N^$(round(alpha, digits=3))) * T1 ^ $(round(mean_gamma, digits=3))")
    println("  Δ(T1, N)       ≈ ($(round(B0, digits=4)) * N^$(round(beta, digits=3))) * T1 ^ $(round(mean_delta, digits=3))")
    
    println("\nFinal MCMC Variance Bounds Equation:")
    println("  Var_max/min(T1, N) = Var_med(T1, N) ± Δ(T1, N)")
end

# Analyze N=6 through N=9 (N=10 takes significantly longer for multiple runs)
derive_generalized_phenomenology(6:9)

--- EXTRACTING PARAMETERS ACROSS SIZES ---
Analyzing N = 6...
Analyzing N = 7...
Analyzing N = 8...
Analyzing N = 9...

=== GENERALIZED N-DEPENDENT PHENOMENOLOGICAL RELATION ===
Averaged Time Exponents (expected ~2.0):
  γ (gamma) = 1.6747
  δ (delta) = 0.8345

Scaling of Prefactors with System Size N:
  A(N) = 1.697 * N ^ -0.7095
  B(N) = 660.8487 * N ^ -3.0399

General Phenomenological Bounds for any N:
  Var_med(T1, N) ≈ (1.697 * N^-0.71) * T1 ^ 1.675
  Δ(T1, N)       ≈ (660.8487 * N^-3.04) * T1 ^ 0.835

Final MCMC Variance Bounds Equation:
  Var_max/min(T1, N) = Var_med(T1, N) ± Δ(T1, N)


# Monte-Carlo

In [9]:
using Logging
disable_logging(Logging.Warn) # Mutes the Plots.jl tick warnings

ENV["GKSwstype"] = "100"
using Plots
using Random
using LinearAlgebra
using Statistics

figures_dir = figure_dir("moments", "monte_carlo", "scaling")
mkpath(figures_dir)

function calculate_MFPT(b::Vector{Float64})
    N_minus_1 = length(b)
    tau = zeros(Float64, N_minus_1)
    tau[1] = 1.0 / b[1]
    for m in 2:N_minus_1
        x_m = (1.0 - b[m]) / b[m]
        tau[m] = (1.0 / b[m]) + x_m * tau[m-1]
    end
    return sum(tau)
end

function calculate_second_moment(b::Vector{Float64}, d::Vector{Float64})
    n_minus_1 = length(b)
    ratios = d ./ b
    inner_sums = zeros(Float64, n_minus_1)
    inner_sums[1] = 1.0 / b[1]
    for k in 2:n_minus_1
        inner_sums[k] = 1.0 / b[k] + ratios[k] * inner_sums[k - 1]
    end
    suffix_sums = zeros(Float64, n_minus_1 + 1)
    for index in n_minus_1:-1:1
        suffix_sums[index] = suffix_sums[index + 1] + inner_sums[index]
    end
    second_sum = 0.0
    for m in 1:n_minus_1
        running_prod = 1.0
        for i in m:-1:1
            if i < m
                running_prod *= ratios[i+1]
            end
            second_sum += (1.0 / b[i]) * suffix_sums[i] * running_prod
        end
    end
    return 2.0 * second_sum - calculate_MFPT(b)
end

function extract_parameters_mc(N::Int, n_samples::Int=10000, seed::Int=42)
    Random.seed!(seed)
    base_b = rand(N - 1)
    
    mfpt_values = zeros(n_samples)
    variance_values = zeros(n_samples)
    
    for s in 1:n_samples
        b = shuffle(base_b)
        t1 = calculate_MFPT(b)
        v = calculate_second_moment(b, 1.0 .- b) - t1^2
        mfpt_values[s] = t1
        variance_values[s] = v
    end
    
    # Filter out any numerical artifacts (variance must be > 0)
    valid_mask = (mfpt_values .> 0) .& (variance_values .> 0)
    mfpt_values = mfpt_values[valid_mask]
    variance_values = variance_values[valid_mask]
    
    log_mfpt = log10.(mfpt_values)
    
    # FIX: Use quantiles to ignore extreme outliers that ruin bin spacing
    min_log_mfpt = quantile(log_mfpt, 0.01)
    max_log_mfpt = quantile(log_mfpt, 0.99)
    num_bins = 40
    bins = range(min_log_mfpt, max_log_mfpt, length=num_bins+1)
    
    bin_centers = Float64[]
    med_vars = Float64[]
    spreads = Float64[]

    for i in 1:num_bins
        if i == num_bins
            in_bin = (log_mfpt .>= bins[i]) .& (log_mfpt .<= bins[i+1])
        else
            in_bin = (log_mfpt .>= bins[i]) .& (log_mfpt .< bins[i+1])
        end
        
        if sum(in_bin) > 5
            v_bin = variance_values[in_bin]
            med_v = median(v_bin)
            spread = (maximum(v_bin) - minimum(v_bin)) / 2.0
            
            if spread > 0
                push!(bin_centers, 10^((bins[i] + bins[i+1]) / 2.0))
                push!(med_vars, med_v)
                push!(spreads, spread)
            end
        end
    end

    # Fallback to ensure we always have enough bins to perform a regression
    mid_log = median(log10.(bin_centers))
    valid = findall(x -> log10(x) >= mid_log, bin_centers)
    if length(valid) < 3
        valid = 1:length(bin_centers)
    end
    
    log_T1 = log10.(bin_centers[valid])
    log_Vmed = log10.(med_vars[valid])
    log_spread = log10.(spreads[valid])
    
    X = hcat(ones(length(log_T1)), log_T1)
    
    params_med = X \ log_Vmed
    A = 10^params_med[1]
    gamma = params_med[2]
    pred_med = X * params_med
    r2_med = 1 - sum((log_Vmed .- pred_med).^2) / sum((log_Vmed .- mean(log_Vmed)).^2)
    
    params_spread = X \ log_spread
    B = 10^params_spread[1]
    delta = params_spread[2]
    pred_spread = X * params_spread
    r2_spread = 1 - sum((log_spread .- pred_spread).^2) / sum((log_spread .- mean(log_spread)).^2)
    
    # FIX: Initialize plot explicitly to guarantee backend flush
    p = plot(title="Phase Space & Phenomenological Fits (N=$N)", grid=true, legend=:topleft,
             xlabel="MFPT, <T1>", ylabel="Variance, Var(T1)", xscale=:log10, yscale=:log10)
             
    scatter!(p, mfpt_values, variance_values, 
        color=:gray, alpha=0.3, markersize=2, markerstrokewidth=0, label="MC Samples"
    )
    
    T1_theory = 10 .^ range(minimum(log_mfpt), maximum(log_mfpt), length=200)
    Var_med_theory = A .* T1_theory .^ gamma
    Spread_theory = B .* T1_theory .^ delta
    
    Var_max_theory = Var_med_theory .+ Spread_theory
    Var_min_theory = Var_med_theory .- Spread_theory
    
    plot!(p, T1_theory, Var_med_theory, color=:blue, linewidth=2, label="Median Fit (γ=$(round(gamma, digits=2)))")
    plot!(p, T1_theory, Var_max_theory, color=:black, linewidth=2, linestyle=:dash, label="Upper Bound")
    plot!(p, T1_theory, Var_min_theory, color=:black, linewidth=2, linestyle=:dash, label="Lower Bound")
    
    savefig(p, joinpath(figures_dir, "mc_fit_N_$N.pdf"))
    
    return A, gamma, r2_med, B, delta, r2_spread
end

function derive_robust_phenomenology(Ns=10:10:100, n_samples=10000)
    As, Bs, gammas, deltas = Float64[], Float64[], Float64[], Float64[]
    
    println("--- EXTRACTING ASYMPTOTIC PARAMETERS ACROSS SIZES ---")
    for N in Ns
        println("Sampling N = $N...")
        A, gamma, r2_med, B, delta, r2_spread = extract_parameters_mc(N, n_samples)
        push!(As, A); push!(Bs, B); push!(gammas, gamma); push!(deltas, delta)
    end
    
    mean_gamma = mean(gammas)
    mean_delta = mean(deltas)
    
    log_Ns = log10.(Float64.(Ns))
    X_N = hcat(ones(length(log_Ns)), log_Ns)
    
    params_A = X_N \ log10.(As)
    A0 = 10^params_A[1]
    alpha = params_A[2]
    r2_A = 1 - sum((log10.(As) .- (X_N * params_A)).^2) / sum((log10.(As) .- mean(log10.(As))).^2)
    
    params_B = X_N \ log10.(Bs)
    B0 = 10^params_B[1]
    beta = params_B[2]
    r2_B = 1 - sum((log10.(Bs) .- (X_N * params_B)).^2) / sum((log10.(Bs) .- mean(log10.(Bs))).^2)
    
    p_scaling = scatter(
        Ns, As, 
        xscale=:log10, yscale=:log10, 
        color=:blue, markersize=5, label="Extracted A(N)", 
        xlabel="System Size (N)", ylabel="Prefactor Values", 
        title="Scaling of Phenomenological Prefactors with N", grid=true, legend=:bottomleft
    )
    scatter!(p_scaling, Ns, Bs, color=:red, markersize=5, label="Extracted B(N)")
    
    N_theory = 10 .^ range(log10(minimum(Ns)), log10(maximum(Ns)), length=100)
    plot!(p_scaling, N_theory, A0 .* N_theory .^ alpha, color=:blue, linewidth=2, linestyle=:dash, label="A(N) Fit (α=$(round(alpha, digits=2)), R²=$(round(r2_A, digits=3)))")
    plot!(p_scaling, N_theory, N_theory .^ beta .* B0, color=:red, linewidth=2, linestyle=:dash, label="B(N) Fit (β=$(round(beta, digits=2)), R²=$(round(r2_B, digits=3)))")
    
    savefig(p_scaling, joinpath(figures_dir, "parameter_scaling_vs_N.pdf"))
    
    println("\n=== TRUE N-DEPENDENT SCALING LAWS ===")
    println("Time Exponents (Averaged across N = $(minimum(Ns)) to $(maximum(Ns))):")
    println("  γ ≈ $(round(mean_gamma, digits=4))")
    println("  δ ≈ $(round(mean_delta, digits=4))")
    
    println("\nSystem Size (N) Scaling:")
    println("  A(N) = $(round(A0, digits=4)) * N ^ $(round(alpha, digits=4)) (R² = $(round(r2_A, digits=4)))")
    println("  B(N) = $(round(B0, digits=4)) * N ^ $(round(beta, digits=4)) (R² = $(round(r2_B, digits=4)))")
end

derive_robust_phenomenology(10:10:500, 10000)

--- EXTRACTING ASYMPTOTIC PARAMETERS ACROSS SIZES ---
Sampling N = 10...
Sampling N = 20...
Sampling N = 30...
Sampling N = 40...
Sampling N = 50...
Sampling N = 60...
Sampling N = 70...
Sampling N = 80...
Sampling N = 90...
Sampling N = 100...
Sampling N = 110...
Sampling N = 120...
Sampling N = 130...
Sampling N = 140...
Sampling N = 150...
Sampling N = 160...
Sampling N = 170...
Sampling N = 180...
Sampling N = 190...
Sampling N = 200...
Sampling N = 210...
Sampling N = 220...
Sampling N = 230...
Sampling N = 240...
Sampling N = 250...
Sampling N = 260...
Sampling N = 270...
Sampling N = 280...
Sampling N = 290...
Sampling N = 300...
Sampling N = 310...
Sampling N = 320...
Sampling N = 330...
Sampling N = 340...
Sampling N = 350...
Sampling N = 360...
Sampling N = 370...
Sampling N = 380...
Sampling N = 390...
Sampling N = 400...
Sampling N = 410...
Sampling N = 420...
Sampling N = 430...
Sampling N = 440...
Sampling N = 450...
Sampling N = 460...
Sampling N = 470...
Sampling N = 48